In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.system(
    "rsync -azPL yalab_dev@biden.tau.ac.il:/home/yalab_dev/yalab-devops/linked_sessions.csv ~/Downloads"
)

sessions = pd.read_csv(
    Path.home() / "Downloads" / "linked_sessions.csv", dtype={"subject_code": str, "session_id": str}
)

receiving incremental file list
linked_sessions.csv
      1,156,857 100%  157.61MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [2]:
ATLAS = "4S456Parcels"
qsiparc_path = Path("/mnt/62/Processed_Data/derivatives/qsiparc")
qsirecon_path = Path("/mnt/62/Processed_Data/derivatives/qsirecon")
parcels = pd.read_csv(qsirecon_path / "atlases" / f"atlas-{ATLAS}"/ f"atlas-{ATLAS}_dseg.tsv", sep="\t")
atlas_nifti = (
    qsirecon_path
    / "atlases"
    / f"atlas-{ATLAS}"
    / f"atlas-{ATLAS}_space-MNI152NLin2009cAsym_res-01_dseg.nii.gz"
)


In [3]:
def parse_file_entities(file_path):
    """Parse BIDS file entities from a file path."""
    entities = {}
    parts = Path(file_path).name.split("_")
    for part in parts:
        if "-" in part:
            key, value = part.split("-", 1)
            entities[key] = value
    return entities

In [5]:
from tqdm import tqdm

recon_workflows = [workflow.name for workflow in qsiparc_path.iterdir() if workflow.name.startswith("qsirecon-")]

data = {}

for workflow in recon_workflows:
    if "AMICONODDI" not in workflow:
        continue
    data[workflow.replace("qsirecon-", "")] = pd.DataFrame()
    for i, row in tqdm(sessions.iterrows(), total=len(sessions), desc=f"Processing workflow {workflow}"):
        subject_code = row["subject_code"]
        session_id = row["session_id"]
        qsiparc_session_path = qsiparc_path / workflow / f"sub-{subject_code}" / f"ses-{session_id}" / "dwi" / f"atlas-{ATLAS}"
        if qsiparc_session_path.exists():
            for fname in qsiparc_session_path.glob("*tsv"):
                entities = parse_file_entities(str(fname))
                df = pd.read_csv(fname, sep="\t")
                df[row.index] = row.values
                df["workflow"] = workflow.replace("qsirecon-", "")
                df["model"] = entities.get("model", "unknown")
                df["param"] = entities.get("param", "unknown")
                df["desc"] = entities.get("desc", "unknown")
                data[workflow.replace("qsirecon-", "")] = pd.concat([data[workflow.replace("qsirecon-", "")], df], ignore_index=True)
                # print(f"Processed: {fname}")
        # else:
                
        #     break
    # break
        


    

Processing workflow qsirecon-AMICONODDI: 100%|██████████| 4429/4429 [3:35:19<00:00,  2.92s/it]   


In [6]:
data["AMICONODDI"]

,index,label,network_label,label_7network,index_17network,label_17network,network_label_17network,atlas_name,network_id,volume_mm3,...,PrivacyStatement,UID,session_id,subject_code,dicom_path,match_type,workflow,model,param,desc
0,1,LH_Vis_1,Vis,7Networks_LH_Vis_1,61.0,17Networks_LH_DorsAttnA_TempOcc_2,DorsAttnA,4S456,NaN,2809.856133,...,Folder not found,S003506,202509161422,AGN29,/mnt/62/Raw_Data/20250916_1422,exact,AMICONODDI,noddi,icvf,unknown
1,2,LH_Vis_2,Vis,7Networks_LH_Vis_2,193.0,17Networks_LH_DefaultC_PHC_2,DefaultC,4S456,NaN,3506.176167,...,Folder not found,S003506,202509161422,AGN29,/mnt/62/Raw_Data/20250916_1422,exact,AMICONODDI,noddi,icvf,unknown
2,3,LH_Vis_3,Vis,7Networks_LH_Vis_3,1.0,17Networks_LH_VisCent_ExStr_1,VisCent,4S456,NaN,2547.712121,...,Folder not found,S003506,202509161422,AGN29,/mnt/62/Raw_Data/20250916_1422,exact,AMICONODDI,noddi,icvf,unknown
3,4,LH_Vis_4,Vis,7Networks_LH_Vis_4,13.0,17Networks_LH_VisPeri_ExStrInf_1,VisPeri,4S456,NaN,3375.104160,...,Folder not found,S003506,202509161422,AGN29,/mnt/62/Raw_Data/20250916_1422,exact,AMICONODDI,noddi,icvf,unknown
4,5,LH_Vis_5,Vis,7Networks_LH_Vis_5,2.0,17Networks_LH_VisCent_ExStr_2,VisCent,4S456,NaN,3366.912160,...,Folder not found,S003506,202509161422,AGN29,/mnt/62/Raw_Data/20250916_1422,exact,AMICONODDI,noddi,icvf,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4271803,452,Cerebellar_Region6,NaN,NaN,NaN,NaN,NaN,Cerebellum,NaN,31059.969475,...,"11/11/2018: No, 11/11/2018: No",S000326,201811111233,BAL01,NaN,missing,AMICONODDI,noddi,tf,unknown
4271804,453,Cerebellar_Region7,NaN,NaN,NaN,NaN,NaN,Cerebellum,NaN,12009.472570,...,"11/11/2018: No, 11/11/2018: No",S000326,201811111233,BAL01,NaN,missing,AMICONODDI,noddi,tf,unknown
4271805,454,Cerebellar_Region8,NaN,NaN,NaN,NaN,NaN,Cerebellum,NaN,16830.464799,...,"11/11/2018: No, 11/11/2018: No",S000326,201811111233,BAL01,NaN,missing,AMICONODDI,noddi,tf,unknown
4271806,455,Cerebellar_Region9,NaN,NaN,NaN,NaN,NaN,Cerebellum,NaN,8548.352406,...,"11/11/2018: No, 11/11/2018: No",S000326,201811111233,BAL01,NaN,missing,AMICONODDI,noddi,tf,unknown
